In [1]:
import os
project_root = r'c:\Projects\traffic-congestion-prediction'
os.chdir(project_root)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import joblib
import time

print("Libraries imported successfully!")
print("Working directory:", os.getcwd())

Libraries imported successfully!
Working directory: c:\Projects\traffic-congestion-prediction


In [2]:
# Load best model and scaler
model = joblib.load('models/xgboost_best.pkl')
scaler = joblib.load('models/scaler.pkl')
class_weights = joblib.load('models/class_weights.pkl')

# Load test data for simulation
X_test = pd.read_csv('data/processed/X_test.csv')
y_test = pd.read_csv(
    'data/processed/y_test.csv').squeeze()

# Feature names
feature_names = X_test.columns.tolist()

print("Model and data loaded successfully!")
print(f"Model type: {type(model).__name__}")
print(f"Features: {feature_names}")
print(f"Test samples: {len(X_test):,}")

Model and data loaded successfully!
Model type: XGBClassifier
Features: ['road_id', 'hour', 'weekday', 'is_weekend', 'is_rush_hour', 'speed', 'speed_lag_1', 'speed_lag_2', 'speed_lag_3', 'speed_lag_6', 'speed_rolling_mean_3', 'speed_rolling_std_3']
Test samples: 278,079


In [6]:
# Load validation data
X_val = pd.read_csv('data/processed/X_val.csv')
y_val = pd.read_csv(
    'data/processed/y_val.csv').squeeze()

# Get probabilities on validation set
y_val_prob = model.predict_proba(X_val)[:, 1]

print("Threshold analysis on VALIDATION set:")
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 35)

for t in [0.35, 0.40, 0.45, 0.50, 0.55, 
          0.60, 0.65, 0.70, 0.75]:
    pred = (y_val_prob > t).astype(int)
    p = precision_score(y_val, pred)
    r = recall_score(y_val, pred)
    print(f"{t:>10.2f} {p:>10.4f} {r:>10.4f}")

Threshold analysis on VALIDATION set:
 Threshold  Precision     Recall
-----------------------------------
      0.35     0.5381     0.9114
      0.40     0.5714     0.8916
      0.45     0.6017     0.8702
      0.50     0.6315     0.8473
      0.55     0.6603     0.8246
      0.60     0.6884     0.7991
      0.65     0.7161     0.7703
      0.70     0.7431     0.7378
      0.75     0.7696     0.6991


In [3]:
# Early warning thresholds
THRESHOLD_GREEN  = 0.40  # below this = no warning
THRESHOLD_YELLOW = 0.65  # below this = caution
THRESHOLD_RED    = 0.65  # above this = warning

def predict_congestion(features_dict):
    """
    Takes a dictionary of traffic features
    Returns probability and warning level
    """
    # Create dataframe from features
    df = pd.DataFrame([features_dict])
    
    # Get probability
    prob = model.predict_proba(df)[0][1]
    
    # Determine warning level
    if prob < THRESHOLD_GREEN:
        level = "GREEN"
        message = "No congestion expected"
        emoji = "🟢"
    elif prob < THRESHOLD_YELLOW:
        level = "YELLOW"
        message = "Congestion possible – monitor"
        emoji = "🟡"
    else:
        level = "RED"
        message = "WARNING: Congestion likely in 20 min"
        emoji = "🔴"
    
    return {
        'probability': prob,
        'level': level,
        'message': message,
        'emoji': emoji
    }

print("Early warning function defined!")
print(f"\nThresholds:")
print(f"GREEN:  probability < {THRESHOLD_GREEN}")
print(f"YELLOW: {THRESHOLD_GREEN} <= probability < {THRESHOLD_YELLOW}")
print(f"RED:    probability >= {THRESHOLD_RED}")

Early warning function defined!

Thresholds:
GREEN:  probability < 0.4
YELLOW: 0.4 <= probability < 0.65
RED:    probability >= 0.65


In [4]:
print("=== Early Warning System – Test Cases ===\n")

# Scenario 1 – Free flowing traffic
scenario1 = {
    'road_id': 0.5,        # normalized
    'hour': 0.125,         # 3am
    'weekday': 0.0,
    'is_weekend': 1.0,
    'is_rush_hour': 0.0,
    'speed': 0.8,          # high speed
    'speed_lag_1': 0.82,
    'speed_lag_2': 0.79,
    'speed_lag_3': 0.81,
    'speed_lag_6': 0.78,
    'speed_rolling_mean_3': 0.80,
    'speed_rolling_std_3': 0.02
}

# Scenario 2 – Rush hour, slowing down
scenario2 = {
    'road_id': 0.5,
    'hour': 0.75,          # 18:00
    'weekday': 0.2,
    'is_weekend': 0.0,
    'is_rush_hour': 1.0,
    'speed': 0.35,         # moderate speed
    'speed_lag_1': 0.42,
    'speed_lag_2': 0.50,
    'speed_lag_3': 0.55,
    'speed_lag_6': 0.60,
    'speed_rolling_mean_3': 0.42,
    'speed_rolling_std_3': 0.08
}

# Scenario 3 – Heavy congestion developing
scenario3 = {
    'road_id': 0.5,
    'hour': 0.75,          # 18:00
    'weekday': 0.2,
    'is_weekend': 0.0,
    'is_rush_hour': 1.0,
    'speed': 0.15,         # very slow
    'speed_lag_1': 0.20,
    'speed_lag_2': 0.30,
    'speed_lag_3': 0.35,
    'speed_lag_6': 0.45,
    'speed_rolling_mean_3': 0.22,
    'speed_rolling_std_3': 0.12
}

for i, (name, scenario) in enumerate([
    ("Free flowing traffic (3am weekend)", scenario1),
    ("Rush hour – slowing down (6pm weekday)", scenario2),
    ("Heavy congestion developing", scenario3)
], 1):
    result = predict_congestion(scenario)
    print(f"Scenario {i}: {name}")
    print(f"  {result['emoji']} {result['level']}: "
          f"{result['message']}")
    print(f"  Congestion probability: "
          f"{result['probability']:.1%}")
    print()

=== Early Warning System – Test Cases ===

Scenario 1: Free flowing traffic (3am weekend)
  🟢 GREEN: No congestion expected
  Congestion probability: 0.2%

Scenario 2: Rush hour – slowing down (6pm weekday)
  🔴 RED: WARNING: Congestion likely in 20 min
  Congestion probability: 78.2%

Scenario 3: Heavy congestion developing
  🔴 RED: WARNING: Congestion likely in 20 min
  Congestion probability: 98.2%



In [5]:
from sklearn.metrics import precision_score, recall_score

y_test_prob = model.predict_proba(X_test)[:, 1]

print("Threshold analysis for warning zones:")
print(f"{'Threshold':>10} {'Precision':>10} {'Recall':>10}")
print("-" * 35)
for t in [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]:
    pred = (y_test_prob > t).astype(int)
    p = precision_score(y_test, pred)
    r = recall_score(y_test, pred)
    print(f"{t:>10.2f} {p:>10.4f} {r:>10.4f}")

Threshold analysis for warning zones:
 Threshold  Precision     Recall
-----------------------------------
      0.50     0.6691     0.8594
      0.55     0.6939     0.8382
      0.60     0.7182     0.8147
      0.65     0.7416     0.7877
      0.70     0.7659     0.7567
      0.75     0.7902     0.7192
